# ScreenLot Modeling Workbook

This notebook explains the current ScreenLot modeling stack: the open-data pipeline, offline ranking evaluation, validation-based model selection, and the explainable hybrid recommender that powers the app.

## What this notebook covers

- loading the processed ScreenLot catalog
- loading the MovieLens 32M ratings sample used for training
- inspecting the saved model card and benchmark leaderboard
- understanding why the live artifact is serving the selected model

In [ ]:
from pathlib import Path

import pandas as pd

from screenlot.model_summary import load_model_card, model_snapshot
from screenlot.modeling import load_processed_catalog, load_ratings_frame
from screenlot.runtime import MODEL_ARTIFACT_PATH, MODEL_CARD_PATH

catalog = load_processed_catalog()
ratings = load_ratings_frame(max_rows=250_000)
model_card = load_model_card()
snapshot = model_snapshot(model_card)

print('Artifact path:', MODEL_ARTIFACT_PATH)
print('Model card path:', MODEL_CARD_PATH)
print('Catalog rows:', len(catalog))
print('Ratings sample rows:', len(ratings))
snapshot

In [ ]:
test_benchmarks = pd.DataFrame((model_card or {}).get('test_benchmarks', []))
validation_benchmarks = pd.DataFrame((model_card or {}).get('validation_benchmarks', []))

display(validation_benchmarks.sort_values(['ndcg@20', 'recall@20'], ascending=False))
display(test_benchmarks.sort_values(['ndcg@20', 'recall@20'], ascending=False))

## Notes

The training script behind this notebook is `scripts/train_screenlot_models.py`. It creates timestamp-aware train/validation/test splits, benchmarks multiple candidate recommenders, tunes the hybrid scorer on validation, and persists the model card plus the serving artifact used by the Streamlit app.